# 🃏 Arte das cartas v3 — consertando rosto, olhos e mãos

Os três problemas da v2 tinham três causas diferentes:

| Problema | Causa | Conserto |
|---|---|---|
| **Sempre o mesmo rosto** | O IP-Adapter comum copia o *conteúdo* da referência, não só o estilo | Modo **só-estilo** (célula 6) |
| **Olhos bugados** | Num corpo inteiro de 832px, o rosto tem ~60px — não cabe olho bom | **Hires fix + conserto de rosto** (célula 8) |
| **Armas/mãos tortas** | Ponto fraco conhecido do SDXL, nenhuma config resolve 100% | Poses que escondem a mão + gerar 3 e escolher |

Rode na ordem. As células 6 e 8 são as novas.

`Ambiente de execução` → `Alterar o tipo` → **GPU T4**

## 1. GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Sem GPU! Ambiente de execução → Alterar tipo → GPU T4"
print("\nGPU ok:", torch.cuda.get_device_name(0))

## 2. Instalar

In [ ]:
!pip -q install --upgrade diffusers transformers accelerate safetensors peft mediapipe compel
print("pronto ✅")

## 3. Onde salvar

In [ ]:
import os

SALVAR_NO_DRIVE = True

if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_SAIDA = '/content/drive/MyDrive/card-game-arte'
else:
    PASTA_SAIDA = '/content/card-game-arte'

os.makedirs(PASTA_SAIDA, exist_ok=True)
print("Salvando em:", PASTA_SAIDA)

## 4. Modelo

| Preset | Cara | Anatomia | Velocidade |
|---|---|---|---|
| `pintado` | fantasia pintada, perto do Hearthstone | média | ~25s |
| `ilustrado` | ilustração colorida | média | ~25s |
| `anime` | anime forte, lado Yu-Gi-Oh | **melhor com rostos** | ~25s |
| `rapido` | turbo, pra iterar prompt | pior | **~6s** ⚡ |

💡 Se rosto é o seu maior problema, teste o `anime`: modelos de anime erram muito menos
olho, porque o estilo já é estilizado e o treino tem milhões de rostos desenhados.

In [ ]:
import torch, gc
from diffusers import (StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline,
                       AutoencoderKL, DPMSolverMultistepScheduler)

PRESETS = {
    "pintado":   dict(id="misri/zavychromaxl_v80", extra="", steps=30, cfg=6.0),
    "ilustrado": dict(id="Lykon/dreamshaper-xl-1-0", extra="", steps=30, cfg=6.0),
    "anime":     dict(id="cagliostrolab/animagine-xl-4.0",
                      extra="masterpiece, high score, great score, absurdres, ", steps=28, cfg=5.0),
    "rapido":    dict(id="Lykon/dreamshaper-xl-v2-turbo", extra="", steps=8, cfg=2.0),
}

ESCOLHA = "pintado"      # <<<<<< pintado | ilustrado | anime | rapido

P = PRESETS[ESCOLHA]
MODELO, EXTRA_QUALIDADE = P["id"], P["extra"]
STEPS, CFG = P["steps"], P["cfg"]
PASSOS_CONSERTO = 10 if ESCOLHA == "rapido" else 25

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)

def carregar(nome):
    try:
        return StableDiffusionXLPipeline.from_pretrained(
            nome, vae=vae, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    except Exception:
        print("(sem variante fp16 nesse repo, baixando a padrão...)")
        return StableDiffusionXLPipeline.from_pretrained(
            nome, vae=vae, torch_dtype=torch.float16, use_safetensors=True)

try:
    del pipe, refino; gc.collect(); torch.cuda.empty_cache()
except NameError:
    pass

pipe = carregar(MODELO)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, algorithm_type="dpmsolver++", use_karras_sigmas=True)
pipe = pipe.to("cuda")

# ⚠️ nada de enable_attention_slicing() — quebra o IP-Adapter da célula 6
pipe.enable_vae_slicing()

# img2img reaproveitando os MESMOS pesos (não gasta VRAM a mais)
refino = StableDiffusionXLImg2ImgPipeline(**pipe.components)
refino.enable_vae_tiling()

gc.collect(); torch.cuda.empty_cache()
print(f"\n✅ preset '{ESCOLHA}' → {MODELO}   |   steps={STEPS}  cfg={CFG}")

## 5. Estilo

Mudança importante: **`upper body`** entrou no estilo. Num corpo inteiro o rosto ocupa
uns 60 pixels e é impossível o modelo acertar o olho. Da cintura pra cima, o rosto ganha
3× mais pixels — e some metade dos problemas de mão junto.

In [ ]:
ESTILO = ("2d digital illustration, hand painted, stylized fantasy card art, "
          "upper body portrait, bold outlines, cel shaded, vivid colors, "
          "dramatic rim light, simple background, dark vignette")

NEGATIVO = ("photo, photograph, photorealistic, realistic, hyperrealistic, dslr, cosplay, "
            "3d render, octane render, cgi, text, watermark, signature, ui, border, frame, "
            "blurry, lowres, bad anatomy, bad hands, deformed hands, malformed fingers, "
            "extra fingers, fused fingers, extra limbs, extra arms, "
            "deformed eyes, asymmetric eyes, crossed eyes, extra pupils, "
            "floating weapon, broken sword, duplicated weapon, "
            "cropped head, out of frame")

LARGURA, ALTURA    = 832, 1216
IMAGENS_POR_PROMPT = 3      # gere 3 e fique com a melhor — é assim que se lida com mãos

print("Estilo configurado ✅")

## 5b. ⭐ Prompts longos (Compel)

O CLIP do SDXL só lê **77 tokens** — o resto é jogado fora silenciosamente (é aquele aviso
`99 > 77` que apareceu no console). Com descrição de rosto + estilo, todo prompt estoura isso.

O **Compel** quebra o prompt em pedaços de 77, codifica cada um e concatena. Resultado:
prompt do tamanho que você quiser, sem perder o final.

In [ ]:
USAR_COMPEL = True

_compel = None
if USAR_COMPEL:
    try:
        from compel import Compel, ReturnedEmbeddingsType
        _compel = Compel(
            tokenizer=[pipe.tokenizer, pipe.tokenizer_2],
            text_encoder=[pipe.text_encoder, pipe.text_encoder_2],
            returned_embeddings_type=ReturnedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED,
            requires_pooled=[False, True],
            truncate_long_prompts=False)
        print("Compel ativo - prompt longo liberado, nada mais e cortado")
    except Exception as e:
        print("Compel indisponivel, prompts serao cortados em 77 tokens:", e)


def emb(prompt):
    """Kwargs de texto pro pipeline: embeddings se o Compel existir, texto puro se nao."""
    if _compel is None:
        return dict(prompt=prompt, negative_prompt=NEGATIVO)
    cond, pooled = _compel(prompt)
    ncond, npooled = _compel(NEGATIVO)
    cond, ncond = _compel.pad_conditioning_tensors_to_same_length([cond, ncond])
    return dict(prompt_embeds=cond, pooled_prompt_embeds=pooled,
                negative_prompt_embeds=ncond, negative_pooled_prompt_embeds=npooled)


print("teste:", list(emb("um teste bem curto").keys()))

## 6. ⭐ Referência — agora em modo SÓ ESTILO

**Aqui estava o bug do "mesmo rosto sempre".** O IP-Adapter injeta a referência em todas as
camadas de atenção do modelo — inclusive as que decidem *quem* é a pessoa. Resultado: ele
copiava o rosto da sua Vanguarda em todas as cartas.

A técnica do **InstantStyle** aplica a referência **só na camada que controla estilo**
(`up_blocks.0`), deixando as camadas de conteúdo livres. Mesma pincelada e paleta,
personagens diferentes.

- `MODO_REFERENCIA = "estilo"` → ✅ recomendado
- `MODO_REFERENCIA = "tudo"` → comportamento antigo (útil só se quiser clonar o personagem)

In [ ]:
import io
from PIL import Image
from IPython.display import display

USAR_REFERENCIA  = True
MODO_REFERENCIA  = "estilo"     # "estilo" | "tudo"
FORCA_REFERENCIA = 0.8          # no modo estilo dá pra usar forte sem clonar o rosto

def aplicar_forca(valor):
    """No modo estilo, mira só a camada de estilo do SDXL (up_blocks.0.attentions.1)."""
    if MODO_REFERENCIA == "estilo":
        try:
            pipe.set_ip_adapter_scale({"up": {"block_0": [0.0, valor, 0.0]}})
            return
        except Exception as e:
            print("(diffusers antigo, caindo pro modo simples enfraquecido)", e)
            valor = valor * 0.45
    pipe.set_ip_adapter_scale(valor)

REF = None
if USAR_REFERENCIA:
    from google.colab import files
    print("Mande UMA imagem de referência de ESTILO:")
    enviados = files.upload()
    nome_ref = list(enviados.keys())[0]
    REF = Image.open(io.BytesIO(enviados[nome_ref])).convert("RGB")
    REF.thumbnail((1024, 1024), Image.LANCZOS)

    try:
        pipe.disable_attention_slicing()
    except Exception:
        pass

    if not getattr(pipe, "_ip_carregado", False):
        pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models",
                             weight_name="ip-adapter_sdxl.bin")
        pipe._ip_carregado = True

    # o img2img foi criado na celula 4, ANTES do IP-Adapter existir - sem registrar o
    # image_encoder nele, hires e conserto de rosto quebram com 'NoneType has no parameters'
    try:
        refino.register_modules(image_encoder=pipe.image_encoder,
                                feature_extractor=pipe.feature_extractor)
    except Exception:
        refino.image_encoder = pipe.image_encoder
        refino.feature_extractor = pipe.feature_extractor

    aplicar_forca(FORCA_REFERENCIA)
    print(f"\n✅ '{nome_ref}' | modo={MODO_REFERENCIA} | força={FORCA_REFERENCIA}")
    display(REF.resize((REF.width // 3, REF.height // 3)))
else:
    try:
        pipe.unload_ip_adapter(); pipe._ip_carregado = False
    except Exception:
        pass
    print("Sem referência — só prompt.")

## 7. Prompts

Use os do `prompts_tank.py` **atualizado** — cada carta agora tem rosto próprio descrito
(idade, cabelo, barba, cicatriz) e vários tanks usam elmo fechado. Isso resolve tanto o
"todo mundo igual" quanto boa parte dos olhos tortos.

In [ ]:
PROMPTS = {

    "tank_t1_vanguarda": "a young clean-shaven human footman with short black hair and a scar "
                         "across his cheek, worn iron armor, round shield strapped to his forearm",

    "tank_t3_capitao_de_ferro": "an iron captain in blackened plate, closed horned helm hiding "
                                "the face, glowing eyes inside the visor, tattered war cape, embers",

    # cole o resto aqui

}

print(f"{len(PROMPTS)} prompt(s) × {IMAGENS_POR_PROMPT} = {len(PROMPTS)*IMAGENS_POR_PROMPT} imagens")

## 8. ⭐ Gerar — com hires fix e conserto de rosto

Cada imagem passa por até três etapas:

1. **Geração** normal em 832×1216
2. **Hires fix** — reamplia 1.4× e repinta levemente. É aqui que textura, olhos e dedos
   ganham resolução de verdade
3. **Conserto de rosto** — acha o rosto, recorta, repinta em 1024×1024 e cola de volta com
   borda suave. É o mesmo princípio do ADetailer do Automatic1111

Custo: ~3× mais lento (uns 80s por imagem no preset `pintado`). Se quiser velocidade pra
testar prompt, desligue os dois e ligue depois só nas cartas aprovadas.

In [ ]:
import random, os, cv2, numpy as np, torch
from PIL import Image, ImageDraw, ImageFilter

HIRES_FIX       = True
ESCALA_HIRES    = 1.4
FORCA_HIRES     = 0.35

CONSERTAR_ROSTO = True
FORCA_ROSTO     = 0.40      # 0.3 = retoque leve | 0.5 = redesenha o rosto


def _kw_ref():
    return {"ip_adapter_image": REF} if REF is not None else {}


def detectar_rosto(img):
    """Devolve (x, y, w, h) do maior rosto, ou None."""
    arr = np.array(img)
    try:
        import mediapipe as mp
        with mp.solutions.face_detection.FaceDetection(
                model_selection=1, min_detection_confidence=0.25) as fd:
            r = fd.process(arr)
            if r.detections:
                b = r.detections[0].location_data.relative_bounding_box
                W, H = img.size
                return (int(b.xmin * W), int(b.ymin * H), int(b.width * W), int(b.height * H))
    except Exception:
        pass
    cas = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    faces = cas.detectMultiScale(cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY), 1.1, 5)
    if len(faces):
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        return (int(x), int(y), int(w), int(h))
    return None


def hires(img, prompt, seed):
    w = int(img.width * ESCALA_HIRES) // 8 * 8
    h = int(img.height * ESCALA_HIRES) // 8 * 8
    grande = img.resize((w, h), Image.LANCZOS)
    return refino(**emb(prompt), image=grande,
                  strength=FORCA_HIRES, num_inference_steps=PASSOS_CONSERTO,
                  guidance_scale=CFG,
                  generator=torch.Generator("cuda").manual_seed(seed),
                  **_kw_ref()).images[0]


def consertar_rosto(img, prompt, seed):
    box = detectar_rosto(img)
    if box is None:
        return img, False
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    lado = int(max(w, h) * 1.9)                      # margem em volta do rosto
    x0, y0 = max(0, int(cx - lado / 2)), max(0, int(cy - lado / 2))
    x1, y1 = min(img.width, x0 + lado), min(img.height, y0 + lado)
    if (x1 - x0) < 48 or (y1 - y0) < 48:
        return img, False

    recorte = img.crop((x0, y0, x1, y1))
    tamanho = recorte.size

    novo = refino(**emb(f"close-up of the face, detailed symmetric eyes, {prompt}"),
                  image=recorte.resize((1024, 1024), Image.LANCZOS),
                  strength=FORCA_ROSTO, num_inference_steps=PASSOS_CONSERTO,
                  guidance_scale=CFG,
                  generator=torch.Generator("cuda").manual_seed(seed),
                  **_kw_ref()).images[0].resize(tamanho, Image.LANCZOS)

    pad = max(4, int(min(tamanho) * 0.14))           # máscara com borda suave
    m = Image.new("L", tamanho, 0)
    ImageDraw.Draw(m).rectangle([pad, pad, tamanho[0] - pad, tamanho[1] - pad], fill=255)
    m = m.filter(ImageFilter.GaussianBlur(pad * 0.7))

    saida = img.copy()
    saida.paste(novo, (x0, y0), m)
    return saida, True


def gerar(prompts=None, seeds=None, quantas=None):
    prompts = PROMPTS if prompts is None else prompts
    quantas = IMAGENS_POR_PROMPT if quantas is None else quantas
    feitas = []

    for nome, corpo in prompts.items():
        prompt_final = f"{EXTRA_QUALIDADE}{corpo}, {ESTILO}"
        for i in range(quantas):
            seed = seeds[i % len(seeds)] if seeds else random.randint(0, 2**31 - 1)

            img = pipe(**emb(prompt_final),
                       width=LARGURA, height=ALTURA, num_inference_steps=STEPS,
                       guidance_scale=CFG,
                       generator=torch.Generator("cuda").manual_seed(seed),
                       **_kw_ref()).images[0]
            torch.cuda.empty_cache()

            etapas = "base"
            if HIRES_FIX:
                img = hires(img, prompt_final, seed); etapas += " + hires"
                torch.cuda.empty_cache()
            if CONSERTAR_ROSTO:
                img, achou = consertar_rosto(img, prompt_final, seed)
                etapas += " + rosto" if achou else " (rosto não encontrado)"
                torch.cuda.empty_cache()

            caminho = os.path.join(PASTA_SAIDA, f"{nome}__{ESCOLHA}__seed{seed}.png")
            img.save(caminho)
            feitas.append(caminho)

            print(f"✅ {nome}   seed={seed}   [{etapas}]")
            display(img.resize((img.width // 2, img.height // 2)))

    print(f"\n{len(feitas)} imagem(ns) em {PASTA_SAIDA}")
    return feitas


gerar()

## 9. Consertar uma imagem que você já tem

Achou uma carta ótima só com o rosto ruim? Passe só ela pelo conserto, sem gerar de novo.

In [ ]:
# caminho = f"{PASTA_SAIDA}/tank_t1_vanguarda__pintado__seed123456.png"
# original = Image.open(caminho).convert("RGB")
# corrigida, achou = consertar_rosto(original, f"{EXTRA_QUALIDADE}{PROMPTS['tank_t1_vanguarda']}, {ESTILO}", 42)
# print("rosto encontrado:", achou)
# corrigida.save(caminho.replace('.png', '_rosto.png'))
# display(corrigida.resize((corrigida.width // 2, corrigida.height // 2)))

## 10. Comparar variações de rosto lado a lado

Mesma carta, mesma seed, forças de conserto diferentes — pra você achar seu número.

In [ ]:
nome, corpo = list(PROMPTS.items())[0]
prompt_final = f"{EXTRA_QUALIDADE}{corpo}, {ESTILO}"
seed_fixa = 777

base = pipe(**emb(prompt_final), width=LARGURA, height=ALTURA,
            num_inference_steps=STEPS, guidance_scale=CFG,
            generator=torch.Generator("cuda").manual_seed(seed_fixa), **_kw_ref()).images[0]
print("SEM conserto:"); display(base.resize((base.width // 3, base.height // 3)))

grande = hires(base, prompt_final, seed_fixa)
for f in [0.3, 0.45, 0.6]:
    FORCA_ROSTO = f
    img, achou = consertar_rosto(grande, prompt_final, seed_fixa)
    print(f"conserto de rosto força {f}  (encontrado: {achou})")
    display(img.resize((img.width // 3, img.height // 3)))
    torch.cuda.empty_cache()

---
### 🖐️ A verdade sobre mãos e armas

Nenhuma configuração de SDXL resolve mão segurando arma de forma confiável — é limitação
do modelo, não sua. O que funciona de verdade:

1. **Esconda a mão na pose.** Arma apoiada no ombro, escudo plantado no chão, braços
   cruzados, punho fechado sobre o peito, mão fora do quadro. Os prompts atualizados já
   fazem isso.
2. **Enquadre da cintura pra cima.** Menos membros no quadro = menos coisa pra errar.
3. **Gere 3 e escolha 1.** É o fluxo normal de quem trabalha com isso.
4. **Elmo fechado** em metade das cartas: zero risco de olho torto e fica ótimo em tank.

### Problemas comuns

| Sintoma | Solução |
|---|---|
| Rosto ainda repetido | `MODO_REFERENCIA = "estilo"` (célula 6) e descreva o rosto no prompt da carta |
| "rosto não encontrado" | Normal em elmo fechado e plano muito aberto — não é erro |
| Rosto colado parece um adesivo | Baixe `FORCA_ROSTO` pra 0.3 |
| Hires mudou demais a imagem | Baixe `FORCA_HIRES` pra 0.25 |
| `CUDA out of memory` | `ESCALA_HIRES = 1.25` e `IMAGENS_POR_PROMPT = 1` |
| `SlicedAttnProcessor missing slice_size` | Rode `pipe.disable_attention_slicing()` |
| `'NoneType' object has no attribute 'parameters'` | O img2img está sem o image_encoder — rode a célula 6 de novo, ela registra |
| `Token indices ... 99 > 77` | O Compel (célula 5b) não carregou; rode-a de novo |